# ARC-AGI-3 Combined End-to-End Run

This notebook combines the Causal Dice submission notebook and the TAAF Kaggle harness in one place.
The first section is the submission agent. The second section is the TAAF runner.


# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py` via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Causal Dice agent for the ARC Prize 2026 ARC-AGI-3 competition.

The policy is game-agnostic. It treats actions as interventions, learns a
compact causal transition model online, and uses deterministic loaded dice to
choose among experiments. Dice weight is expected information gain plus
learned progress value, minus death, repetition, cycle, and action costs.
"""
from __future__ import annotations

import hashlib
import math
import random
from collections import Counter, defaultdict, deque
from typing import Any, Optional

import numpy as np
from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent


class Evidence:
    __slots__ = (
        "attempts", "changes", "no_changes", "deaths", "progress",
        "delta_sum", "novelty_sum", "value",
    )

    def __init__(self) -> None:
        self.attempts = 0
        self.changes = 0
        self.no_changes = 0
        self.deaths = 0
        self.progress = 0
        self.delta_sum = 0.0
        self.novelty_sum = 0.0
        self.value = 0.0


class Candidate:
    __slots__ = (
        "action_id", "key", "model_key", "question", "target",
        "target_label", "target_prior", "utility", "information", "risk",
        "confidence",
    )

    def __init__(
        self,
        action_id: int,
        key: tuple[Any, ...],
        model_key: tuple[Any, ...],
        question: str,
        target: Optional[tuple[int, int]] = None,
        target_label: str = "",
        target_prior: float = 0.0,
    ) -> None:
        self.action_id = action_id
        self.key = key
        self.model_key = model_key
        self.question = question
        self.target = target
        self.target_label = target_label
        self.target_prior = target_prior
        self.utility = 0.0
        self.information = 0.0
        self.risk = 0.0
        self.confidence = 0.0


class Pending:
    __slots__ = ("state_signature", "grid", "levels_completed", "candidate")

    def __init__(
        self,
        state_signature: str,
        grid: np.ndarray,
        levels_completed: int,
        candidate: Candidate,
    ) -> None:
        self.state_signature = state_signature
        self.grid = grid
        self.levels_completed = levels_completed
        self.candidate = candidate


class TraceEntry:
    __slots__ = ("state_signature", "candidate", "changed", "reward")

    def __init__(
        self,
        state_signature: str,
        candidate: Candidate,
        changed: bool,
        reward: float,
    ) -> None:
        self.state_signature = state_signature
        self.candidate = candidate
        self.changed = changed
        self.reward = reward


class MyAgent(Agent):
    """Online causal explorer with deterministic, uncertainty-aware dice."""

    MAX_ACTIONS = 400
    MAX_COMPONENTS = 96
    TRACE_CREDIT_DEPTH = 36

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        digest = hashlib.sha256(self.game_id.encode("utf-8")).digest()
        self.seed = int.from_bytes(digest[:8], "big", signed=False)
        self.rng = random.Random(self.seed)

        self.step_index = 0
        self.current_level = 0
        self.level_start_step = 0
        self.pending: Optional[Pending] = None
        self.recent_change_mask: Optional[np.ndarray] = None
        self.last_effective_action_id: Optional[int] = None
        self.controlled_center: Optional[tuple[int, int]] = None
        self.walkable_colors: Counter[int] = Counter()
        self.navigation_bonus: dict[int, float] = {}

        self.evidence: defaultdict[tuple[Any, ...], Evidence] = defaultdict(Evidence)
        self.state_q: defaultdict[tuple[str, tuple[Any, ...]], float] = defaultdict(float)
        self.role_q: defaultdict[tuple[Any, ...], float] = defaultdict(float)
        self.state_visits: Counter[str] = Counter()
        self.state_action_visits: Counter[tuple[str, tuple[Any, ...]]] = Counter()
        self.transitions: defaultdict[
            tuple[str, tuple[Any, ...]], Counter[str]
        ] = defaultdict(Counter)
        self.trace: deque[TraceEntry] = deque(maxlen=256)

        self.success_macro: list[tuple[int, tuple[Any, ...]]] = []
        self.macro_cursor = 0
        self.progress_events = 0
        self.death_events = 0

    @property
    def name(self) -> str:
        return f"{super().name}.causal-dice-v1.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    @staticmethod
    def _grid_from(frame: FrameData) -> np.ndarray:
        if not frame.frame:
            return np.zeros((1, 1), dtype=np.uint8)
        raw = np.asarray(frame.frame[-1])
        if raw.ndim != 2 or raw.size == 0:
            return np.zeros((1, 1), dtype=np.uint8)
        return np.clip(raw, 0, 255).astype(np.uint8, copy=False)

    @staticmethod
    def _signature(grid: np.ndarray) -> str:
        h = hashlib.blake2b(digest_size=12)
        h.update(int(grid.shape[0]).to_bytes(2, "big"))
        h.update(int(grid.shape[1]).to_bytes(2, "big"))
        h.update(grid.tobytes(order="C"))
        return h.hexdigest()

    @staticmethod
    def _levels(frame: FrameData) -> int:
        try:
            return int(frame.levels_completed)
        except (TypeError, ValueError):
            return 0

    @staticmethod
    def _legal_action_ids(frame: FrameData) -> list[int]:
        ids: list[int] = []
        for raw in frame.available_actions or []:
            try:
                value = int(raw.value if isinstance(raw, GameAction) else raw)
            except (AttributeError, TypeError, ValueError):
                continue
            if 1 <= value <= 7 and value not in ids:
                ids.append(value)
        if not ids:
            ids = [a.value for a in GameAction if a is not GameAction.RESET]
        return sorted(ids)

    @staticmethod
    def _aligned_delta(before: np.ndarray, after: np.ndarray) -> tuple[float, np.ndarray]:
        if before.shape == after.shape:
            mask = before != after
            return float(mask.mean()), mask
        height = max(before.shape[0], after.shape[0])
        width = max(before.shape[1], after.shape[1])
        old = np.full((height, width), 255, dtype=np.uint8)
        new = np.full((height, width), 254, dtype=np.uint8)
        old[: before.shape[0], : before.shape[1]] = before
        new[: after.shape[0], : after.shape[1]] = after
        mask = old != new
        return float(mask.mean()), mask

    def _reset_level_local(self, level: int) -> None:
        self.current_level = level
        self.level_start_step = self.step_index
        self.state_visits.clear()
        self.state_action_visits.clear()
        self.transitions.clear()
        self.state_q.clear()
        self.trace.clear()
        self.recent_change_mask = None
        self.macro_cursor = 0

    def _credit_progress(self) -> None:
        useful = [entry for entry in self.trace if entry.changed]
        suffix = useful[-20:]
        if suffix:
            self.success_macro = [
                (entry.candidate.action_id, entry.candidate.model_key)
                for entry in suffix
            ]
        for distance, entry in enumerate(reversed(list(self.trace)[-self.TRACE_CREDIT_DEPTH:])):
            credit = 12.0 * (0.86**distance)
            state_key = (entry.state_signature, entry.candidate.key)
            self.state_q[state_key] = min(20.0, self.state_q[state_key] + credit)
            self.role_q[entry.candidate.model_key] = min(
                12.0, self.role_q[entry.candidate.model_key] + 0.35 * credit
            )

    def _punish_failure_trace(self) -> None:
        for distance, entry in enumerate(reversed(list(self.trace)[-10:])):
            penalty = 4.0 * (0.72**distance)
            state_key = (entry.state_signature, entry.candidate.key)
            self.state_q[state_key] = max(-12.0, self.state_q[state_key] - penalty)
            self.role_q[entry.candidate.model_key] = max(
                -8.0, self.role_q[entry.candidate.model_key] - 0.2 * penalty
            )

    def _observe_pending(self, grid: np.ndarray, latest_frame: FrameData) -> None:
        if self.pending is None:
            return

        pending = self.pending
        candidate = pending.candidate
        delta, mask = self._aligned_delta(pending.grid, grid)
        changed = bool(delta > 0.0)
        levels = self._levels(latest_frame)
        progressed = bool(
            levels > pending.levels_completed or latest_frame.state is GameState.WIN
        )
        died = latest_frame.state is GameState.GAME_OVER
        new_signature = self._signature(grid)
        novel = self.state_visits[new_signature] == 0

        ev = self.evidence[candidate.key]
        ev.attempts += 1
        ev.changes += int(changed)
        ev.no_changes += int(not changed)
        ev.deaths += int(died)
        ev.progress += int(progressed)
        ev.delta_sum += delta
        ev.novelty_sum += float(novel)

        reward = -0.08
        reward += min(0.7, delta * 5.0)
        reward += 0.35 if changed and novel else 0.0
        reward += 25.0 if progressed else 0.0
        reward -= 8.0 if died else 0.0
        ev.value = 0.82 * ev.value + 0.18 * reward

        state_key = (pending.state_signature, candidate.key)
        old_q = self.state_q[state_key]
        self.state_q[state_key] = max(-15.0, min(25.0, old_q + 0.35 * (reward - old_q)))
        old_role = self.role_q[candidate.model_key]
        self.role_q[candidate.model_key] = max(
            -10.0, min(15.0, old_role + 0.12 * (reward - old_role))
        )
        self.transitions[state_key][new_signature] += 1
        self.trace.append(TraceEntry(pending.state_signature, candidate, changed, reward))
        self.recent_change_mask = mask
        self.last_effective_action_id = candidate.action_id if changed and not died else None
        if changed and not died and 1 <= candidate.action_id <= 4:
            self._learn_motion(pending.grid, grid)

        if progressed:
            self.progress_events += 1
            self._credit_progress()
        if died:
            self.death_events += 1
            self._punish_failure_trace()

        self.pending = None

    def _learn_motion(self, before: np.ndarray, after: np.ndarray) -> None:
        if before.shape != after.shape:
            return
        mask = before != after
        if not mask.any():
            return
        combined = np.concatenate((before[mask], after[mask]))
        values, counts = np.unique(combined, return_counts=True)
        terrain = int(values[int(np.argmax(counts))])
        new_object = mask & (after != terrain)
        if int(new_object.sum()) == 0:
            return
        ys, xs = np.nonzero(new_object)
        self.controlled_center = (
            int(round(float(xs.mean()))),
            int(round(float(ys.mean()))),
        )
        self.walkable_colors[terrain] += 1

    @staticmethod
    def _bucket(value: float, edges: tuple[float, ...]) -> int:
        return sum(value >= edge for edge in edges)

    def _components(self, grid: np.ndarray) -> list[dict[str, Any]]:
        height, width = grid.shape
        values, counts = np.unique(grid, return_counts=True)
        if len(values) == 0:
            return []
        background = int(values[int(np.argmax(counts))])
        frequency = {int(v): int(c) for v, c in zip(values, counts)}
        seen = np.zeros_like(grid, dtype=bool)
        components: list[dict[str, Any]] = []
        total = max(1, grid.size)

        for y0 in range(height):
            for x0 in range(width):
                color = int(grid[y0, x0])
                if color == background or seen[y0, x0]:
                    continue
                stack = [(y0, x0)]
                seen[y0, x0] = True
                cells: list[tuple[int, int]] = []
                while stack:
                    y, x = stack.pop()
                    cells.append((y, x))
                    for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                        if (
                            0 <= ny < height
                            and 0 <= nx < width
                            and not seen[ny, nx]
                            and int(grid[ny, nx]) == color
                        ):
                            seen[ny, nx] = True
                            stack.append((ny, nx))

                ys = [p[0] for p in cells]
                xs = [p[1] for p in cells]
                y1, y2 = min(ys), max(ys)
                x1, x2 = min(xs), max(xs)
                box_area = max(1, (y2 - y1 + 1) * (x2 - x1 + 1))
                area = len(cells)
                if area > int(0.55 * total):
                    continue
                density = area / box_area
                aspect = (x2 - x1 + 1) / max(1, y2 - y1 + 1)
                border = int(x1 == 0 or y1 == 0 or x2 == width - 1 or y2 == height - 1)
                color_rarity = 1.0 - frequency[color] / total
                smallness = 1.0 / math.sqrt(max(1, area))
                change_overlap = 0.0
                if self.recent_change_mask is not None and self.recent_change_mask.shape == grid.shape:
                    change_overlap = sum(bool(self.recent_change_mask[y, x]) for y, x in cells) / area
                mean_x = sum(xs) / area
                mean_y = sum(ys) / area
                center_y, center_x = min(
                    cells,
                    key=lambda point: (
                        (point[1] - mean_x) ** 2 + (point[0] - mean_y) ** 2,
                        point[0],
                        point[1],
                    ),
                )
                descriptor = (
                    "object",
                    color,
                    self._bucket(float(area), (2, 4, 9, 17, 33, 65, 129)),
                    self._bucket(aspect, (0.5, 0.8, 1.25, 2.0)),
                    self._bucket(density, (0.35, 0.7, 0.95)),
                    border,
                )
                score = 1.4 * color_rarity + 0.8 * smallness + 0.5 * density + 0.8 * change_overlap
                components.append(
                    {
                        "descriptor": descriptor,
                        "x": int(center_x),
                        "y": int(center_y),
                        "area": area,
                        "color": color,
                        "score": float(score),
                        "cells": cells,
                    }
                )

        components.sort(key=lambda item: (-item["score"], item["area"], item["y"], item["x"]))
        return components[: self.MAX_COMPONENTS]

    @staticmethod
    def _spatial_probes(grid: np.ndarray) -> list[tuple[int, int, str]]:
        height, width = grid.shape
        fractions = (0.125, 0.375, 0.625, 0.875)
        probes: list[tuple[int, int, str]] = []
        for row, fy in enumerate(fractions):
            for column, fx in enumerate(fractions):
                x = max(0, min(63, min(width - 1, int(round(fx * (width - 1))))))
                y = max(0, min(63, min(height - 1, int(round(fy * (height - 1))))))
                probes.append((x, y, f"region-{row}-{column}"))
        return probes

    def _navigation_preferences(self, grid: np.ndarray) -> dict[int, float]:
        if self.controlled_center is None or not self.walkable_colors:
            return {}
        terrain = self.walkable_colors.most_common(1)[0][0]
        traversable = grid == terrain
        height, width = grid.shape
        center_x, center_y = self.controlled_center

        starts: list[tuple[int, int, int]] = []
        for y in range(max(0, center_y - 6), min(height, center_y + 7)):
            for x in range(max(0, center_x - 6), min(width, center_x + 7)):
                if traversable[y, x]:
                    starts.append((abs(x - center_x) + abs(y - center_y), y, x))
        if not starts:
            return {}
        _, start_y, start_x = min(starts)
        start = (start_y, start_x)

        queue: deque[tuple[int, int]] = deque([start])
        distance: dict[tuple[int, int], int] = {start: 0}
        parent: dict[tuple[int, int], tuple[int, int]] = {}
        while queue:
            y, x = queue.popleft()
            for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                point = (ny, nx)
                if (
                    0 <= ny < height
                    and 0 <= nx < width
                    and traversable[ny, nx]
                    and point not in distance
                ):
                    distance[point] = distance[(y, x)] + 1
                    parent[point] = (y, x)
                    queue.append(point)

        targets: list[tuple[float, tuple[int, int]]] = []
        for component in self._components(grid):
            if component["color"] == terrain:
                continue
            if (component["x"] - center_x) ** 2 + (component["y"] - center_y) ** 2 <= 64:
                continue
            approaches: list[tuple[int, tuple[int, int]]] = []
            for y, x in component["cells"]:
                for point in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if point in distance:
                        approaches.append((distance[point], point))
            if not approaches:
                continue
            path_distance, approach = min(approaches)
            salience = float(component["score"])
            salience += 0.015 * min(80, int(component["area"]))
            salience -= 0.006 * path_distance
            targets.append((salience, approach))
        if not targets:
            return {}

        _, goal = max(targets, key=lambda item: (item[0], -distance[item[1]]))
        cursor = goal
        while parent.get(cursor) is not None and parent[cursor] != start:
            cursor = parent[cursor]
        dy = cursor[0] - start_y
        dx = cursor[1] - start_x
        if abs(dx) > abs(dy):
            action_id = 4 if dx > 0 else 3
        elif dy != 0:
            action_id = 2 if dy > 0 else 1
        elif dx != 0:
            action_id = 4 if dx > 0 else 3
        else:
            return {}
        return {action_id: 0.7}

    def _candidate_set(
        self, grid: np.ndarray, signature: str, latest_frame: FrameData
    ) -> list[Candidate]:
        legal = self._legal_action_ids(latest_frame)
        self.navigation_bonus = self._navigation_preferences(grid)
        candidates: list[Candidate] = []
        for action_id in legal:
            if action_id == 6:
                components = self._components(grid)
                occupied: set[tuple[int, int]] = set()
                for component in components:
                    descriptor = component["descriptor"]
                    x = max(0, min(63, int(component["x"])))
                    y = max(0, min(63, int(component["y"])))
                    occupied.add((x, y))
                    label = f"color={component['color']},area={component['area']}"
                    candidates.append(
                        Candidate(
                            action_id=6,
                            key=(6, descriptor, x, y),
                            model_key=("click", descriptor),
                            question="object-role",
                            target=(x, y),
                            target_label=label,
                            target_prior=float(component["score"]),
                        )
                    )
                for x, y, label in self._spatial_probes(grid):
                    if (x, y) in occupied:
                        continue
                    descriptor = ("region", label)
                    candidates.append(
                        Candidate(
                            action_id=6,
                            key=(6, descriptor, int(x), int(y)),
                            model_key=("click", descriptor),
                            question="spatial-intervention",
                            target=(int(x), int(y)),
                            target_label=label,
                            target_prior=0.08,
                        )
                    )
            else:
                question = "undo-causality" if action_id == 7 else "action-effect"
                candidates.append(
                    Candidate(
                        action_id=action_id,
                        key=(action_id,),
                        model_key=("simple", action_id),
                        question=question,
                    )
                )
        return candidates

    def _predicted_state(self, signature: str, key: tuple[Any, ...]) -> Optional[str]:
        counter = self.transitions.get((signature, key))
        if not counter:
            return None
        return counter.most_common(1)[0][0]

    def _score_candidates(self, candidates: list[Candidate], signature: str) -> None:
        for candidate in candidates:
            ev = self.evidence[candidate.key]
            exact_count = self.state_action_visits[(signature, candidate.key)]
            change_probability = (ev.changes + 1.0) / (ev.attempts + 2.0)
            progress_probability = (ev.progress + 0.05) / (ev.attempts + 2.0)
            death_probability = (ev.deaths + 0.15) / (ev.attempts + 2.0)
            information = 1.0 / math.sqrt(ev.attempts + exact_count + 1.0)
            predicted = self._predicted_state(signature, candidate.key)
            predicted_visits = self.state_visits[predicted] if predicted is not None else 0
            frontier = 1.25 if predicted is None else 1.0 / math.sqrt(predicted_visits + 1.0)
            no_effect_penalty = 0.0
            if ev.attempts >= 2:
                no_effect_penalty = 1.8 * (ev.no_changes / ev.attempts)
            repeat_penalty = 0.9 * math.log1p(exact_count)
            cycle_penalty = 0.55 * math.log1p(predicted_visits)
            undo_penalty = 0.65 if candidate.action_id == 7 and exact_count == 0 else 0.0

            macro_bonus = 0.0
            if self.macro_cursor < len(self.success_macro):
                wanted_action, wanted_model = self.success_macro[self.macro_cursor]
                if candidate.action_id == wanted_action and candidate.model_key == wanted_model:
                    macro_bonus = 2.2

            inertia_bonus = 0.0
            if candidate.action_id == self.last_effective_action_id and candidate.action_id <= 5:
                inertia_bonus = 0.45
            navigation = self.navigation_bonus.get(candidate.action_id, 0.0)
            if ev.attempts >= 2 and ev.no_changes / ev.attempts > 0.5:
                navigation = 0.0

            learned = 0.55 * self.state_q[(signature, candidate.key)]
            learned += 0.28 * self.role_q[candidate.model_key]
            utility = (
                learned
                + 3.0 * progress_probability
                + 0.75 * change_probability
                + 1.35 * information
                + frontier
                + 0.35 * candidate.target_prior
                + macro_bonus
                + inertia_bonus
                + navigation
                - 4.5 * death_probability
                - no_effect_penalty
                - repeat_penalty
                - cycle_penalty
                - undo_penalty
                - 0.12
            )
            candidate.utility = float(utility)
            candidate.information = float(information)
            candidate.risk = float(death_probability)
            candidate.confidence = float(
                max(0.0, min(1.0, 1.0 - information * 0.7 - death_probability * 0.3))
            )

    def _loaded_die(self, candidates: list[Candidate]) -> tuple[Candidate, str, float]:
        ordered = sorted(
            candidates,
            key=lambda c: (-c.utility, c.action_id, c.target or (-1, -1)),
        )
        best = ordered[0]
        gap = best.utility - ordered[1].utility if len(ordered) > 1 else math.inf
        best_ev = self.evidence[best.key]

        if best_ev.progress > 0 or (best_ev.attempts >= 3 and gap >= 1.25):
            return best, "exploit", 1.0

        elapsed = max(0, self.step_index - self.level_start_step)
        temperature = max(0.18, 1.05 * math.exp(-elapsed / 140.0))
        selected = best
        selected_roll = -math.inf
        for candidate in ordered:
            uniform = min(1.0 - 1e-12, max(1e-12, self.rng.random()))
            gumbel = -math.log(-math.log(uniform))
            roll = candidate.utility + temperature * gumbel
            if roll > selected_roll:
                selected = candidate
                selected_roll = roll
        return selected, "explore", float(selected_roll)

    @staticmethod
    def _json_key(value: tuple[Any, ...]) -> str:
        return "/".join(str(part) for part in value)

    def _materialize_action(
        self,
        selected: Candidate,
        mode: str,
        die_roll: float,
        candidates: list[Candidate],
    ) -> GameAction:
        action = GameAction.from_id(int(selected.action_id))
        if selected.target is not None:
            x, y = selected.target
            action.set_data({"x": int(x), "y": int(y)})
        else:
            action.set_data({})

        alternatives = sorted(candidates, key=lambda c: -c.utility)[:6]
        action.reasoning = {
            "policy": "causal-dice-v1",
            "mode": mode,
            "question": selected.question,
            "selected": action.name,
            "target": list(selected.target) if selected.target is not None else None,
            "target_label": selected.target_label or None,
            "utility": round(selected.utility, 5),
            "information_gain": round(selected.information, 5),
            "estimated_risk": round(selected.risk, 5),
            "confidence": round(selected.confidence, 5),
            "die_roll": round(die_roll, 5),
            "alternatives": [
                {
                    "action": f"ACTION{c.action_id}",
                    "target": list(c.target) if c.target is not None else None,
                    "hypothesis": self._json_key(c.model_key),
                    "utility": round(c.utility, 5),
                }
                for c in alternatives
            ],
            "observed_progress_events": int(self.progress_events),
            "observed_failures": int(self.death_events),
        }
        return action

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        grid = self._grid_from(latest_frame)
        self._observe_pending(grid, latest_frame)
        levels = self._levels(latest_frame)

        if levels != self.current_level:
            self._reset_level_local(levels)

        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            if latest_frame.state is GameState.GAME_OVER:
                self.trace.clear()
                self.macro_cursor = 0
            reset = GameAction.RESET
            reset.set_data({})
            reset.reasoning = {
                "policy": "causal-dice-v1",
                "mode": "required-reset",
                "reason": latest_frame.state.value,
            }
            return reset

        signature = self._signature(grid)
        self.state_visits[signature] += 1
        candidates = self._candidate_set(grid, signature, latest_frame)
        if not candidates:
            fallback_id = self._legal_action_ids(latest_frame)[0]
            candidates = [
                Candidate(
                    action_id=fallback_id,
                    key=(fallback_id,),
                    model_key=("simple", fallback_id),
                    question="legal-fallback",
                )
            ]

        self._score_candidates(candidates, signature)
        selected, mode, die_roll = self._loaded_die(candidates)
        self.state_action_visits[(signature, selected.key)] += 1
        if self.macro_cursor < len(self.success_macro):
            wanted_action, wanted_model = self.success_macro[self.macro_cursor]
            if selected.action_id == wanted_action and selected.model_key == wanted_model:
                self.macro_cursor += 1

        action = self._materialize_action(selected, mode, die_roll, candidates)
        self.pending = Pending(signature, grid.copy(), levels, selected)
        self.step_index += 1
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()


# TAAF Harness

The cells below keep the TAAF runner available in the same notebook for end-to-end execution and debugging.


# TAAF ARC-AGI-3 Kaggle Run

This notebook is generated by the Tufa ARC-AGI Framework (TAAF), an open-source deployment harness from [Tufa Labs](https://tufalabs.ai/) for running ARC-AGI-3 solvers reproducibly on Kaggle.

The notebook installs the ARC runtime, makes the bundled TAAF source snapshot importable, runs any solver setup commands, loads the pickled benchmark, and writes results to `/kaggle/working`. It can run as a public/offline debug notebook or as the same code path used for competition reruns. Kaggle's `KAGGLE_IS_COMPETITION_RERUN` flag always wins and switches the run into submission mode.

For quick inline experiments, use the customization code cell just before the benchmark run. That is the safest place to tweak the benchmark or solver after the deployed bundle has loaded.


In [1]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

TAAF RUN_AS_SUBMISSION=False
taaf.kaggle: LIBRARY_PATH=/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs


In [2]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
DATASET_SOURCES: list[str] = ["jakobbrggen/taaf-kaggle-source-anim-20260807-anim", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES: list[str] = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

TAAF source bundle: /kaggle/input/datasets/jakobbrggen/taaf-kaggle-source-anim-20260807-anim
taaf.kaggle: input paths = {"driessmit1/arc3-vllm-h100-wheelhouse-v3": "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": "/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot", "jakobbrggen/taaf-kaggle-source-anim-20260807-anim": "/kaggle/input/datasets/jakobbrggen/taaf-kaggle-source-anim-20260807-anim"}


In [4]:
# Audit attached datasets
import subprocess

subprocess.run(["ls", "/kaggle/input"], check=False)

competitions
datasets


CompletedProcess(args=['ls', '/kaggle/input'], returncode=0)

In [5]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return
    commands = json.loads(path.read_text(encoding="utf-8"))
    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(str(command), shell=True, check=check, cwd=WORKING_DIR, env=env)
        if not check and result.returncode != 0:
            print(f"taaf.kaggle: {label} command exited with {result.returncode}", flush=True)
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Run deployment setup commands before the benchmark pickle is loaded.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

taaf.kaggle: wrote /usr/local/lib/python3.12/dist-packages/taaf_kaggle_sources.pth (3 source roots)
taaf.kaggle: setup command: "$PYTHON" - <<'PYSETUP'
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

WHEELHOUSE_OWNER = 'driessmit1'
WHEELHOUSE_SLUG = 'arc3-vllm-h100-wheelhouse-v3'
MODEL_OWNER = 'driessmit1'
MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'
SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 1234
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'
VLLM_MAX_MODEL_LEN = 65536
ANALYZER_CONTEXT_WINDOW = 32768
VLLM_TENSOR_PARALLEL_SIZE = 1
WORKING_DIR = Path(os.environ['TAAF_KAGGLE_WORKING_DIR'])
SITE_PACKAGES = WORKING_DIR / 'vllm-site-packages'
VLLM_SERVER_LOG = WORKING_DIR / 'vllm-openai-server.log'
VLLM_SERVER_PID = WORKING_DIR / 'vllm-openai-server.pid'
INSTALL_STAMP = SITE_PACKAGES / f'.{WHEELHOUSE_SLUG}'
STAMP_TEXT = 'vllm==0.19.0 torch==2.10.0 flashinfer==0.6.6\n'


In [6]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [7]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [8]:
# Inline customization hook.
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
# Example:
# bm.label = f'{bm.label}-debug'


In [9]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

benchmark.label : anim-20260807-anim
benchmark.solver: HarnessSolver(label='duck-harness', runtime_environment=None, job_dir=None, soft_end_time=None, minimal_diagnostics=False, model='local', analyzer_timeout=900.0, max_actions_per_game=None, max_runtime_s_per_game=7920.0, concurrency=28, save_request_logs=False, hard_noop_guard=True, animation_awareness=True, start_local_server=False, local_server_config='', local_server_CREDENTIAL_REDACTED='/Users/jakobbruggen/Desktop/duck-harness/ARC3-Inference', local_server_port=None, local_server_tensor_parallel_size=None, local_server_count=1, cancel_drain_timeout_s=120.0)
benchmark.passes: 4
benchmark.games : 6
git status:
  ARC3-Inference                   9158303     DIRTY  feature/animation-awareness  feat: proactively suggest animation retrieval when stuck (s…
  tufa-arc-agi-framework           9158303     DIRTY  feature/animation-awareness  feat: proactively suggest animation retrieval when stuck (s…
deploy.kaggle: working_dir            

analyzer request failed at action 48: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=900.0)


benchmark: regenerated diagnostics in /kaggle/working in 5.21s
benchmark: anim-20260807-anim
solver:    duck-harness
games:     6
passes:    4
runs:      24 (won: 0)
started:   2026-08-07 16:09:28
ended:     in progress
mean score:    3.09
median score:  0.70
total actions: 3452
total tokens:  1538865
generated tokens/sec: 196.62 (job wallclock)
total wallclock: 182456.2s

per-game (mean across passes):
  bp35-0a0ad940: score=0.39, levels=1.0/9, actions=327, tokens=63834
  ft09-0d8bbf25: score=9.29, levels=1.5/6, actions=100, tokens=63490
  g50t-5849a774: score=0.89, levels=0.2/7, actions=54, tokens=63303
  r11l-495a7899: score=3.52, levels=1.0/6, actions=69, tokens=65381
  sb26-7fbdac44: score=4.46, levels=1.5/8, actions=136, tokens=63845
  sk48-d8078629: score=0.00, levels=0.0/8, actions=175, tokens=64861



analyzer request failed at action 200: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=26.51054817899967)
analyzer request failed at action 44: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=21.743645692999053)
analyzer request failed at action 64: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=577.9839829620014)
analyzer request failed at action 45: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=121.38741349200063)
analyzer request failed at action 48: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=159.70187153400002)
analyzer request failed at action 138: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=72.20061808900027)
analyzer request failed at action 143: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=77.52993657600018)
analyzer request failed at action 430: HTT

[finished] ft09-0d8bbf25 state=gave_up level=0/6 score=0.00 actions=199 tokens=66476 per-level=199/43,0/12,0/23,0/28,0/65,0/37 note="tokens=66476"
[finished] r11l-495a7899 state=gave_up level=1/6 score=4.76 actions=43 tokens=66706 per-level=7/22,36/33,0/51,0/26,0/52,0/49 note="tokens=67301"
[finished] ft09-0d8bbf25 state=gave_up level=2/6 score=14.29 actions=44 tokens=66706 per-level=22/43,7/12,15/23,0/28,0/65,0/37 note="tokens=66706"
[finished] ft09-0d8bbf25 state=gave_up level=2/6 score=8.59 actions=47 tokens=55847 per-level=26/43,21/12,0/23,0/28,0/65,0/37 note="tokens=58436"
[finished] sb26-7fbdac44 state=gave_up level=1/8 score=2.78 actions=63 tokens=59395 per-level=10/18,53/28,0/18,0/19,0/31,0/23,0/58,0/18 note="tokens=62518"
[finished] sk48-d8078629 state=gave_up level=0/8 score=0.00 actions=137 tokens=65321 per-level=137/61,0/177,0/101,0/103,0/230,0/181,0/125,0/92 note="tokens=65839"
[finished] sb26-7fbdac44 state=gave_up level=1/8 score=2.78 actions=142 tokens=65006 per-level=1

...

[finished] bp35-0a0ad940 state=gave_up level=1/9 score=0.30 actions=516 tokens=62095 per-level=57/21,459/48,0/44,0/38,0/33,0/87,0/86,0/131,0/163 note="tokens=62095"
[finished] r11l-495a7899 state=gave_up level=1/6 score=3.69 actions=50 tokens=66696 per-level=25/22,25/33,0/51,0/26,0/52,0/49 note="tokens=67452"
[finished] sb26-7fbdac44 state=gave_up level=1/8 score=2.78 actions=105 tokens=66668 per-level=17/18,88/28,0/18,0/19,0/31,0/23,0/58,0/18 note="tokens=66933"


analyzer request failed at action 39: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=37.18546086299921)


[finished] g50t-5849a774 state=gave_up level=0/7 score=0.00 actions=38 tokens=67327 per-level=38/78,0/175,0/179,0/230,0/96,0/54,0/67 note="tokens=67656"
[finished] sb26-7fbdac44 state=gave_up level=3/8 score=9.52 actions=238 tokens=65716 per-level=15/18,147/28,21/18,55/19,0/31,0/23,0/58,0/18 note="tokens=67663"
[finished] g50t-5849a774 state=gave_up level=1/7 score=3.57 actions=65 tokens=62827 per-level=40/78,25/175,0/179,0/230,0/96,0/54,0/67 note="tokens=68589"


analyzer request failed at action 122: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=305.6429488269987)


[finished] sk48-d8078629 state=gave_up level=0/8 score=0.00 actions=121 tokens=64364 per-level=121/61,0/177,0/101,0/103,0/230,0/181,0/125,0/92 note="tokens=64759"
benchmark: regenerated diagnostics in /kaggle/working in 3.71s
benchmark: anim-20260807-anim
solver:    duck-harness
games:     6
passes:    4
runs:      24 (won: 0)
started:   2026-08-07 16:09:28
ended:     2026-08-07 18:22:15
duration:  2h 12m 47s
mean score:    3.09
median score:  0.70
total actions: 3523
total tokens:  1554765
generated tokens/sec: 195.15 (job wallclock)
total wallclock: 190256.7s

per-game (mean across passes):
  bp35-0a0ad940: score=0.39, levels=1.0/9, actions=334, tokens=64656
  ft09-0d8bbf25: score=9.29, levels=1.5/6, actions=101, tokens=63684
  g50t-5849a774: score=0.89, levels=0.2/7, actions=54, tokens=63802
  r11l-495a7899: score=3.52, levels=1.0/6, actions=71, tokens=66708
  sb26-7fbdac44: score=4.46, levels=1.5/8, actions=137, tokens=64196
  sk48-d8078629: score=0.00, levels=0.0/8, actions=181, t